# Calculate $\gamma_{thermal}$ and $\gamma_{spatial}$

This shows how the thermal and spatial baseline contributions to coherence are calculated, then divided out of the NISAR coherence to leave $\gamma_{temporal}$. 

In [ ]:
import numpy as np
import xarray as xr
import rioxarray  # noqa: F401
import matplotlib.pyplot as plt
from seaborn import set_theme
set_theme()

import sys
import os
from pathlib import Path

# import from this repo 
project_root = str(Path(os.getcwd()).parent)
if project_root not in sys.path:
    sys.path.append(project_root)

%load_ext autoreload
%autoreload 2
from scripts.download import build_snr_timeseries, interpolate_radar_grid_to_dem
from scripts.coherence import thermal_coherence, spatial_coherence

First, we setup logging to get useful information and error messages. 

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    force=True 
)

Open the data cubes that were saved by the download notebook. 

In [ ]:
data_dir = "../data/"

nisar_20m = xr.open_zarr(data_dir + 'nisar/zarrs/obs_20m.zarr')
nisar_80m = xr.open_zarr(data_dir + 'nisar/zarrs/obs_80m.zarr')

Next, load the ASO snow depth and use its footprint to mask the cubes. 

In [ ]:
aso = xr.open_dataarray(data_dir + 'aso/er_20260318/ASO_EastRiver_2026Mar18_snowdepth_3m.tif')

nisar_80m = nisar_80m.where(aso.rio.reproject_match(nisar_80m).notnull())
nisar_20m = nisar_20m.where(aso.rio.reproject_match(nisar_20m).notnull())

The start and end time of each date pair are stored in the cube. 

In [ ]:
starts = nisar_80m['start'].values
ends = nisar_80m['end'].values

## $\gamma_{thermal}$

Thermal noise coherence depends on the SNR of the two acquisitions in each pair. We get the SNR from the backscatter and NEBZ files. 

In [ ]:
bs_files = [item.resolve() for item in Path(data_dir + 'nisar/east_river/').rglob('*backscatter*')]
nebz_files = [item.resolve() for item in Path(data_dir + 'nisar/east_river/').rglob('*NEBZ*')]

Build a time series of SNR from the files. The time zone offset needs to match the cubes. 

In [ ]:
snr = build_snr_timeseries(bs_files, nebz_files, utc_offset_hours=-7)

Resample the SNR to 20 m and mask it to the ASO footprint. 

In [ ]:
res = 20

snr_dwn = snr.rio.reproject(
    snr.rio.crs,
    resolution=res
)
snr_dwn = snr_dwn.where(aso.rio.reproject_match(snr_dwn).notnull())

In [ ]:
snr_dwn.plot(col='date', col_wrap=5, vmin=0, vmax=1000)

Calculate $\gamma_{thermal}$ for every date pair, then plot it. 

In [ ]:
gamma_thermal = thermal_coherence(snr_dwn, starts, ends)
gamma_thermal = gamma_thermal.rio.write_crs(snr.rio.crs)

gamma_thermal.plot(col='pair', col_wrap=5, vmin=0, vmax=1, cmap='inferno')

## $\gamma_{spatial}$

Spatial baseline coherence depends on the perpendicular baseline, incidence angle and slant range. These are interpolated onto the DEM. 

In [ ]:
inc_files = [item.resolve() for item in Path(data_dir + 'nisar/east_river/').rglob('*incidence_angle*')]
bl_files = [item.resolve() for item in Path(data_dir + 'nisar/east_river/').rglob('*perpendicular_baseline*')]
r_files = [item.resolve() for item in Path(data_dir + 'nisar/east_river/').rglob('*reference_slant_range*')]

Interpolate each radar grid layer onto the 80 m DEM. 

In [ ]:
baselines = interpolate_radar_grid_to_dem(nisar_80m['elevation'], bl_files, utc_offset_hours=-7)
thetas = interpolate_radar_grid_to_dem(nisar_80m['elevation'], inc_files, utc_offset_hours=-7)
rs = interpolate_radar_grid_to_dem(nisar_80m['elevation'], r_files, utc_offset_hours=-7)

Calculate $\gamma_{spatial}$, then plot it. 

In [ ]:
res = 3 
wvl = .24
gamma_spatial = spatial_coherence(baselines, thetas, rs, res, wvl)

gamma_spatial.plot(col='pair', col_wrap=5)

## $\gamma_{temporal}$

Dividing the coherence by $\gamma_{thermal}$ and $\gamma_{spatial}$ leaves $\gamma_{temporal}$. 

In [ ]:
nisar_80m['gamma_temporal'] = nisar_80m['coherence'] / (gamma_thermal.rio.reproject_match(nisar_80m) * gamma_spatial.rio.reproject_match(nisar_80m))
nisar_20m['gamma_temporal'] = nisar_20m['coherence'] / (gamma_thermal.rio.reproject_match(nisar_20m) * gamma_spatial.rio.reproject_match(nisar_20m))

Plot $\gamma_{temporal}$ for both grids. 

In [ ]:
nisar_80m['gamma_temporal'].plot(col='pair', col_wrap=5, cmap='inferno')
nisar_20m['gamma_temporal'].plot(col='pair', col_wrap=5, cmap='inferno', vmax=1)

Save the zarrs. 

In [ ]:
snr.to_zarr(data_dir + 'nisar/zarrs/snr.zarr', mode="w")
gamma_thermal.to_dataset(name='gamma_thermal').to_zarr(data_dir + 'nisar/zarrs/gamma_thermal.zarr', mode="w")
gamma_spatial.to_dataset(name='gamma_spatial').to_zarr(data_dir + 'nisar/zarrs/gamma_spatial.zarr', mode="w")
nisar_20m.to_zarr(data_dir + 'nisar/zarrs/coherence_20m.zarr', mode="w")
nisar_80m.to_zarr(data_dir + 'nisar/zarrs/coherence_80m.zarr', mode="w")